# Phase 2: Classical ML Baselines — Analysis

This notebook loads the trained Random Forest and SVM models from Phase 2 and presents:
1. Summary comparison table across all models and tiers
2. Confusion matrices
3. Feature importance analysis
4. ROC curves (Tier 1 binary)
5. Per-class analysis and observations

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

# Project paths
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluate import get_tier_config, compute_metrics
from features import CLASSICAL_FEATURE_NAMES, AGG_STAT_NAMES

MODEL_DIR = PROJECT_ROOT / "models" / "m1_classical"
RESULTS_DIR = PROJECT_ROOT / "results"
DATA_DIR = PROJECT_ROOT / "data"

sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")

## 1. Load Results and Data

In [ ]:
# Load evaluation results
with open(MODEL_DIR / "evaluation_results.json") as f:
    eval_results = json.load(f)

with open(MODEL_DIR / "search_results.json") as f:
    search_results = json.load(f)

# Load summary CSV
summary_df = pd.read_csv(RESULTS_DIR / "classical_ml_summary.csv")
summary_df

In [ ]:
# Load test data for re-computing predictions
def load_test_data(tier):
    """Load test features and labels for a given tier."""
    test = np.load(DATA_DIR / "classical_ml_features" / "test.npz")
    X = test["X"]
    y = test[f"y_tier{tier}"]
    if tier == 3:
        mask = y != -1
        X, y = X[mask], y[mask]
    return X, y

# Load all models
models = {}
for algo in ["rf", "svm"]:
    for tier in [1, 2, 3]:
        key = f"{algo}_tier{tier}"
        models[key] = joblib.load(MODEL_DIR / f"{key}.joblib")

# Also load the reduced-feature RF
models["rf_tier2_top50"] = joblib.load(MODEL_DIR / "rf_tier2_top50.joblib")
top50_idx = np.load(MODEL_DIR / "top50_feature_indices.npy")

print(f"Loaded {len(models)} models")

## 2. Summary Comparison Table

In [ ]:
# Build a formatted comparison table
rows = []
for algo in ["rf", "svm"]:
    for tier in [1, 2, 3]:
        key = f"{algo}_tier{tier}"
        m = eval_results["models"][key]
        cfg = get_tier_config(tier)
        rows.append({
            "Algorithm": "Random Forest" if algo == "rf" else "SVM (RBF/Linear)",
            "Tier": f"Tier {tier} ({cfg['num_classes']}-class)",
            "Accuracy": f"{m['accuracy']:.4f}",
            "F1 Macro": f"{m['f1_macro']:.4f}",
            "F1 Weighted": f"{m['f1_weighted']:.4f}",
            "CV F1 Macro": f"{m['cv_f1_macro']:.4f}",
        })

comparison_df = pd.DataFrame(rows)
comparison_df.style.set_caption("Classical ML Baseline Results — All Models and Tiers")

In [ ]:
# Best hyperparameters for each model
print("Best Hyperparameters")
print("=" * 60)
for key, sr in search_results.items():
    print(f"\n{key}:")
    for param, val in sr["best_params"].items():
        print(f"  {param}: {val}")
    print(f"  CV F1 Macro: {sr['best_cv_f1_macro']:.4f}")

## 3. Confusion Matrices

In [ ]:
# Plot confusion matrices side-by-side: RF vs SVM for each tier
for tier in [1, 2, 3]:
    cfg = get_tier_config(tier)
    X_test, y_test = load_test_data(tier)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5 + len(cfg["class_names"]) * 0.3))
    
    for idx, (algo, name) in enumerate([("rf", "Random Forest"), ("svm", "SVM")]):
        key = f"{algo}_tier{tier}"
        model = models[key]
        y_pred = model.predict(X_test)
        
        cm = confusion_matrix(y_test, y_pred, labels=range(cfg["num_classes"]))
        cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
        
        sns.heatmap(
            cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=cfg["class_names"], yticklabels=cfg["class_names"],
            ax=axes[idx],
        )
        axes[idx].set_xlabel("Predicted")
        axes[idx].set_ylabel("True")
        axes[idx].set_title(f"{name} — Tier {tier} (Normalized)")
    
    fig.tight_layout()
    plt.show()

## 4. Feature Importance Analysis

In [ ]:
# Build full 441 feature names
feature_names = [
    f"{feat}_{stat}"
    for feat in CLASSICAL_FEATURE_NAMES
    for stat in AGG_STAT_NAMES
]

# Tier 2 RF feature importances
rf_tier2 = models["rf_tier2"]
importances = rf_tier2.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

# Top-30 bar chart
top_n = 30
fig, ax = plt.subplots(figsize=(10, 10))
top_idx = sorted_idx[:top_n]
top_names = [feature_names[i] for i in top_idx]
top_values = importances[top_idx]

y_pos = np.arange(top_n)
ax.barh(y_pos, top_values[::-1], color="steelblue")
ax.set_yticks(y_pos)
ax.set_yticklabels(top_names[::-1], fontsize=9)
ax.set_xlabel("Feature Importance")
ax.set_title("Top-30 Feature Importances — RF (Tier 2)")
fig.tight_layout()
plt.show()

In [ ]:
# Group importance by base feature (sum across 7 aggregation stats)
base_importance = {}
for i, imp in enumerate(importances):
    base_feat = CLASSICAL_FEATURE_NAMES[i // 7]
    base_importance[base_feat] = base_importance.get(base_feat, 0.0) + imp

base_df = pd.DataFrame(
    sorted(base_importance.items(), key=lambda x: x[1], reverse=True),
    columns=["Feature", "Total Importance"],
)

fig, ax = plt.subplots(figsize=(10, 10))
top_base = base_df.head(20)
ax.barh(np.arange(len(top_base)), top_base["Total Importance"].values[::-1], color="coral")
ax.set_yticks(np.arange(len(top_base)))
ax.set_yticklabels(top_base["Feature"].values[::-1], fontsize=9)
ax.set_xlabel("Aggregated Importance (sum of 7 stats)")
ax.set_title("Top-20 Base Features by Total Importance — RF (Tier 2)")
fig.tight_layout()
plt.show()

print("Top-10 base features:")
base_df.head(10)

In [ ]:
# Compare full (441 features) vs reduced (top-50) RF on Tier 2
reduced_metrics = eval_results["feature_importance"]["reduced_model_metrics"]
full_metrics = eval_results["models"]["rf_tier2"]

comparison_rows = [
    {"Model": "RF Tier 2 (441 features)", 
     "Accuracy": full_metrics["accuracy"], 
     "F1 Macro": full_metrics["f1_macro"],
     "F1 Weighted": full_metrics["f1_weighted"]},
    {"Model": "RF Tier 2 (top-50 features)", 
     "Accuracy": reduced_metrics["accuracy"], 
     "F1 Macro": reduced_metrics["f1_macro"],
     "F1 Weighted": reduced_metrics["f1_weighted"]},
]

feat_comp_df = pd.DataFrame(comparison_rows)
print("Full vs. Reduced Feature Set (Tier 2 RF):")
feat_comp_df

## 5. ROC Curves (Tier 1 — Binary)

In [ ]:
X_test_t1, y_test_t1 = load_test_data(1)

fig, ax = plt.subplots(figsize=(7, 7))

for algo, name, color in [("rf", "Random Forest", "steelblue"), ("svm", "SVM", "coral")]:
    model = models[f"{algo}_tier1"]
    y_prob = model.predict_proba(X_test_t1)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_t1, y_prob)
    auc_val = roc_auc_score(y_test_t1, y_prob)
    ax.plot(fpr, tpr, lw=2, color=color, label=f"{name} (AUC = {auc_val:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Tier 1 (Normal vs. Anomaly)")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

## 6. Per-Class Analysis

In [ ]:
# Detailed per-class metrics for Tier 2 (primary target)
X_test_t2, y_test_t2 = load_test_data(2)
cfg2 = get_tier_config(2)

print("Tier 2 — Detailed Classification Reports")
print("=" * 60)

for algo, name in [("rf", "Random Forest"), ("svm", "SVM")]:
    model = models[f"{algo}_tier2"]
    y_pred = model.predict(X_test_t2)
    print(f"\n--- {name} ---")
    print(classification_report(
        y_test_t2, y_pred, 
        target_names=cfg2["class_names"], 
        digits=4, zero_division=0,
    ))

In [ ]:
# Per-class F1 comparison bar chart for Tier 2
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(cfg2["num_classes"])
width = 0.35

for i, (algo, name, color) in enumerate([
    ("rf", "Random Forest", "steelblue"), 
    ("svm", "SVM", "coral"),
]):
    model = models[f"{algo}_tier2"]
    y_pred = model.predict(X_test_t2)
    report = classification_report(
        y_test_t2, y_pred, 
        target_names=cfg2["class_names"],
        output_dict=True, zero_division=0,
    )
    f1_per_class = [report[c]["f1-score"] for c in cfg2["class_names"]]
    ax.bar(x + i * width, f1_per_class, width, label=name, color=color)

ax.set_xticks(x + width / 2)
ax.set_xticklabels(cfg2["class_names"], rotation=30, ha="right")
ax.set_ylabel("F1-Score")
ax.set_title("Per-Class F1 Scores — Tier 2")
ax.legend()
ax.set_ylim(0, 1.05)
fig.tight_layout()
plt.show()

## 7. Observations

**Key findings from Phase 2 classical ML baselines:**

1. **Tier 1 (binary):** Both models perform well. SVM achieves 93.3% accuracy and 0.916 F1 macro, slightly outperforming RF (89.9%, 0.872). These set a strong baseline for the binary anomaly detection task.

2. **Tier 2 (6-class, primary target):** Both models achieve ~85% accuracy but macro F1 drops to ~0.68–0.70, indicating weak performance on minority classes. "Normal Start-Up" (only 9 test samples) is the hardest class for both models. The class imbalance (Idle Fault dominates at 118/208 test samples) inflates overall accuracy.

3. **Tier 3 (9-class):** Performance is surprisingly competitive with Tier 2 despite having more classes, likely because splitting fault types into specific conditions creates more distinguishable acoustic patterns. Both models achieve ~0.70–0.71 F1 macro.

4. **Feature importance:** MFCC-related features dominate (especially delta2_mfcc_0 and mfcc_0 statistics), followed by spectral centroid and RMS energy. The reduced 50-feature RF performs slightly worse than the full 441-feature model, suggesting that even lower-ranked features contribute useful signal.

5. **RF vs SVM:** SVM generally edges out RF on Tiers 1–2, while RF is slightly better on Tier 3. The differences are modest. SVM benefits from StandardScaler normalization and the RBF kernel's ability to model non-linear boundaries.

**These baselines establish the performance floor. The neural network models (Phase 3) should improve on these results, particularly for minority classes, through data augmentation and learned feature representations.**